# Fase H · Transferencia horizontal parcial — Mac mini

**Por qué.** Con el protocolo de la Fase 1c, L-SHADE y jSO ganan a MAREA-2 (`sin_colonias`) en 5 de 6 celdas. Por clase de función, frente a jSO:

| celda | unimodales | multimodales | híbridas | composición |
|---|---|---|---|---|
| D10 50k | 0/2/4 (+9,6) | 1/8/15 (+0,1) | 1/1/17 (+1,0) | 4/3/15 (+0,0) |
| D10 500k | 0/2/4 (+5,6) | 1/7/16 (+0,3) | 0/0/19 (+1,9) | 4/4/14 (+0,0) |
| D50 50k | 0/1/5 (+2,8) | 8/4/12 (+0,0) | 1/3/15 (+0,7) | 2/1/19 (+0,1) |
| D50 500k | 0/2/4 (+7,8) | 0/7/17 (+0,2) | 0/0/19 (+2,0) | 2/4/16 (+0,0) |

Victorias/empates/derrotas de MAREA-2 (Wilcoxon rank-sum, α=0,05); entre paréntesis, log10 de mediana MAREA-2 / jSO, mediana por clase.
Las **híbridas** se pierden todas (~100× peor): mezclan funciones base sobre subconjuntos de variables, y MAREA-2 mueve las D coordenadas a la vez.

**Hipótesis.** La conjugación transfiere un plásmido con una *fracción* de los genes. Si el movimiento se aplica solo en un subconjunto de coordenadas, la brecha en híbridas debería cerrarse.

**Brazos** (un cambio respecto a `sin_colonias`, semillas de la ablación 0–14):
- `A2_hgt`: fracción por célula ~ N(µ, 0,1), µ adaptada por EMA hacia las fracciones exitosas (inicio 0,5).
- `A2_hgt05`: fracción fija 0,5 (control: ¿importa la adaptación?).

**Pasos:** 0) entorno y hashes · 1) `sin_colonias` sigue bit a bit (355/355) · 2) 50k · 3) 500k (opcional) · 4) análisis.
Copiar al mini: `marea2.py` y `ablacion_marea2.py` (commit 6272b39), y tener los `LSHADE_*`/`JSO_*` en `resultados_experimento/`.
**Solo en el Mac mini.**

In [ ]:
import sys, os, hashlib, subprocess, opfunu

WORKERS = 20
ESPERADO = {
    "marea2.py":                 "10d7448e59082adf4f464068646d74d25d8b8f54f5317f9eb4dfed32dacfe64b",
    "ablacion_marea2.py":        "651e3ac4c4ee019965f50bdb2519d0902d96ad97dbcf964e38773c5adedb7ef3",
    "verificar_sin_colonias.py": "d1f7667ea3f4a92cc2b5fdc4d5801dbcd01fe7e66bf6c1675e96168a96961ea0",
}
CFGS = ["A2_hgt", "A2_hgt05"]

def correr(*args):
    """Ejecuta un script con el mismo Python del kernel y muestra la salida en vivo."""
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        if "pkg_resources" not in linea:
            print(linea, end="")
    return p.wait()

print("python :", sys.executable)
print("carpeta:", os.getcwd())
print("opfunu :", opfunu.__version__)
assert opfunu.__version__ == "1.0.4", "opfunu tiene que ser 1.0.4 (1.0.1 esta roto)"
for f, h in ESPERADO.items():
    real = hashlib.sha256(open(f, "rb").read()).hexdigest()
    print(f"{f:28s} {real[:16]}  {'OK' if real == h else 'DISTINTO'}")
    assert real == h, f"{f} no es la version del commit 6272b39: copiala de nuevo"
print("\nEntorno y archivos OK")

## 1 · Verificar `sin_colonias` (unos minutos)
`marea2.py` cambió: tiene que seguir diciendo **355/355 corridas identicas bit a bit**.

In [ ]:
rc = correr("verificar_sin_colonias.py", "--runs", "5", "--workers", str(WORKERS))
assert rc == 0, "sin_colonias NO es identico: no seguir, mandar la salida a Claude"
VERIFICADO = True

## 2 · Brazos a 50k (minutos)

In [ ]:
assert globals().get("VERIFICADO"), "primero la celda de verificacion"
rc = correr("correr_marea2.py", "--ablacion", "--cfgs", *CFGS,
            "--maxfes", "50000", "--runs", "15", "--workers", str(WORKERS))
assert rc == 0

## 3 · Brazos a 500k (~6 h, opcional)
Mirar primero el análisis a 50k (celda 4). Si `A2_hgt` no mejora las híbridas a 50k, no vale la pena.
Poner `RUN_500K = True` para correrla.

In [ ]:
RUN_500K = False
assert globals().get("VERIFICADO"), "primero la celda de verificacion"
if RUN_500K:
    rc = correr("correr_marea2.py", "--ablacion", "--cfgs", *CFGS,
                "--maxfes", "500000", "--runs", "15", "--workers", str(WORKERS))
    assert rc == 0
else:
    print("500k omitido (RUN_500K = False)")

## 4 · Análisis (segundos; se puede correr también en el MacBook)
- **vs `sin_colonias`**: mismas semillas → Wilcoxon *signed-rank* pareado. B/E/W del brazo (B = el brazo gana).
- **vs jSO / L-SHADE**: primeras 15 corridas del protocolo, semillas distintas → *rank-sum*. W/T/L del brazo y log10 de la razón de medianas (negativo = el brazo es mejor).
- Se separa **CEC2014** (conjunto de ajuste) de **CEC2017+2022** (reservado): si luego se toca algo, decidir solo con CEC2014.

In [ ]:
import json, math, numpy as np
from scipy.stats import wilcoxon, ranksums
from experimento_config import get_problems_for_dim
from ablacion_marea2 import clase, CLASES

RUNS = 15
def cargar(alg, d, fes):
    for p in (f"resultados_experimento/ablacion/MAREA2_{alg}_D{d}_FES{fes}.json",
              f"resultados_experimento/{alg}_D{d}_FES{fes}.json", f"{alg}_D{d}_FES{fes}.json"):
        if os.path.exists(p):
            M = np.full((71, RUNS), np.nan)
            for k, v in json.load(open(p)).items():
                a, b = k.split("_"); i, r = int(a[3:]), int(b[3:])
                if r < RUNS: M[i, r] = v
            return M if np.isfinite(M).all() else None
    return None

def bew(A, B, sel):          # pareado: A es el brazo, B la referencia
    b = e = w = 0
    for q in sel:
        if np.all(A[q] == B[q]): e += 1; continue
        pv = wilcoxon(A[q], B[q])[1]
        if pv >= 0.05: e += 1
        elif A[q].mean() < B[q].mean(): b += 1
        else: w += 1
    return f"{b}/{e}/{w}"

def wtl(A, B, sel):          # no pareado + log10 razon de medianas
    w = t = l = 0; lr = []
    for q in sel:
        ma, mb = np.median(A[q]), np.median(B[q])
        lr.append(math.log10((ma + 1e-8) / (mb + 1e-8)))
        if np.allclose(A[q], B[q]) or ranksums(A[q], B[q]).pvalue >= 0.05: t += 1
        elif ma < mb: w += 1
        else: l += 1
    return f"{w}/{t}/{l} ({np.median(lr):+.2f})" if sel else "-"

for fes in (50000, 500000):
    for d in (10, 50):
        probs = get_problems_for_dim(d)
        cls = [clase(p["name"], i) for i, p in enumerate(probs)]
        suite = ["2014" if p["name"].endswith("2014") else "otras" for p in probs]
        base = cargar("sin_colonias", d, fes)
        pool = {c: cargar(c, d, fes) for c in CFGS}
        pool = {c: M for c, M in pool.items() if M is not None}
        if base is None or not pool:
            continue
        rivales = {r: M for r in ("JSO", "LSHADE") if (M := cargar(r, d, fes)) is not None}
        print(f"\n{'='*100}\nD{d}  {fes//1000}k")
        for grupo, filtro in (("CEC2014 (ajuste)", "2014"), ("CEC2017+2022 (reservado)", "otras")):
            print(f"\n  {grupo}")
            print(f"  {'brazo':14s}{'vs':14s}" + "".join(f"{c:>20s}" for c in CLASES) + f"{'total':>20s}")
            for c, M in pool.items():
                sel = lambda k: [q for q in range(71) if suite[q] == filtro and (k is None or cls[q] == k)]
                print(f"  {c:14s}{'sin_colonias':14s}" + "".join(f"{bew(M, base, sel(k)):>20s}" for k in CLASES)
                      + f"{bew(M, base, sel(None)):>20s}")
                for r, R in rivales.items():
                    print(f"  {'':14s}{r:14s}" + "".join(f"{wtl(M, R, sel(k)):>20s}" for k in CLASES)
                          + f"{wtl(M, R, sel(None)):>20s}")
            for r, R in rivales.items():   # referencia: donde estaba sin_colonias
                sel = lambda k: [q for q in range(71) if suite[q] == filtro and (k is None or cls[q] == k)]
                print(f"  {'sin_colonias':14s}{r:14s}" + "".join(f"{wtl(base, R, sel(k)):>20s}" for k in CLASES)
                      + f"{wtl(base, R, sel(None)):>20s}")

## 5 · Traer los JSON al MacBook
`resultados_experimento/ablacion/MAREA2_A2_hgt*_D{10,50}_FES{50000,500000}.json` → misma carpeta en el MacBook (no a la raíz).

In [ ]:
ABL = "resultados_experimento/ablacion"
for c in CFGS:
    for fes in (50000, 500000):
        for d in (10, 50):
            p = f"{ABL}/MAREA2_{c}_D{d}_FES{fes}.json"
            if not os.path.exists(p):
                print(f"FALTA   {p}"); continue
            v = json.load(open(p))
            malos = sum(1 for x in v.values() if not math.isfinite(x))
            print(f"{'OK     ' if len(v) == 71*15 and not malos else 'REVISAR'} {os.path.basename(p)}: "
                  f"{len(v)}/{71*15} corridas, {malos} no finitos")